# MOT20 Pedestrian Tracking Benchmark — ByteTrack + YOLO26

**Goal.** Measure how well **ByteTrack** tracks pedestrians in the extremely crowded **MOT20** dataset when paired with two YOLO26 detectors of different sizes (`yolo26s`, `yolo26m`).

### Pipeline
Dataset discovery → YOLO-format conversion → detector benchmark → ByteTrack tracking + system profiling → TrackEval (HOTA / MOTA / IDF1) → qualitative visualizations → conclusions

### Experimental setup
| Item | Value |
| :--- | :--- |
| Dataset | MOT20 train split: MOT20-01, -02, -03, -05 (8,931 frames) |
| Detectors | `yolo26s`, `yolo26m` (pretrained COCO weights, no fine-tuning) |
| Target class | Person only (COCO class 0) |
| Input size | 1280 |
| Detection confidence / NMS IoU | 0.10 / 0.70 |
| Tracker | ByteTrack (Ultralytics `bytetrack.yaml`) |
| Metrics | HOTA, DetA, AssA, MOTA, IDF1, ID Switches, Frag, FPS, latency, GPU memory |
| Hardware | Kaggle Tesla T4 GPU |

### Contents
1. Environment check · 2. Dependencies · 3. Dataset paths & helpers · 4. Shared configuration · 5. YOLO dataset conversion · 6. Detector benchmark · 7. ByteTrack configuration · 8. Tracking & profiling · 9. System metrics · 10. Tracking accuracy (TrackEval) · 11. Visual analysis · 12. Video demos · 13. Conclusions

## Part 1: Environment Check & Dataset File Listing
This standard Kaggle cell walks through `/kaggle/input` and prints every file path. Its only purpose is to confirm that the **MOT20** dataset is attached to the notebook. The listing is long because every video frame is a separate image.

In [ ]:
import numpy as np
import pandas as pd
import os



## Part 2: Installing Dependencies
| Package | Why it is needed |
| :--- | :--- |
| `ultralytics` | YOLO26 detectors and the built-in **ByteTrack** tracker |
| `pycocotools` | COCO-style evaluation utilities used by Ultralytics validation |
| `lap` | Linear-assignment (Hungarian) solver used in the tracker's matching step |
| TrackEval (`git clone`) | Official MOT benchmark code that computes **HOTA, MOTA, IDF1** (used in Part 10) |

> If the clone prints "already exists" on a re-run, that is harmless.

In [ ]:
!pip -q install -U ultralytics pycocotools pandas "lap>=0.5.12"
!git clone -q https://github.com/JonathonLuiten/TrackEval.git /kaggle/working/TrackEval

## Part 3: Dataset Paths & Helper Functions
This cell locates the MOT20 **images** (`MOT20/train`) and **labels** (`MOT20Labels/train`) folders and defines three helpers used by the rest of the notebook:

| Helper | Role |
| :--- | :--- |
| `ground_truth_path(seq)` | Finds `gt/gt.txt` for a sequence |
| `frame_files(seq)` | Returns all frames of a sequence in temporal order |
| `seq_image_size(seq)` | Reads image width/height from `seqinfo.ini` (no need to open every frame) |

**Sanity check:** the printed frame counts should be 429 (MOT20-01), 2,782 (MOT20-02), 2,405 (MOT20-03) and 3,315 (MOT20-05), i.e. 8,931 frames in total.

In [ ]:
import gc
import json
import time
import shutil
import configparser
from pathlib import Path

import torch
from PIL import Image
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/mot20_benchmark")
OUT.mkdir(parents=True, exist_ok=True)

def find_train_folder(parent_name):
    matches = [
        p for p in INPUT.rglob("train")
        if p.is_dir()
        and p.parent.name == parent_name
        and any(p.glob("MOT20-*"))
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {parent_name}/train folder; found: {matches}"
        )
    return matches[0]

IMAGE_TRAIN = find_train_folder("MOT20")
LABEL_TRAIN = find_train_folder("MOT20Labels")

SEQUENCES = sorted(p.name for p in IMAGE_TRAIN.glob("MOT20-*") if p.is_dir())

def ground_truth_path(seq):
    candidates = [
        LABEL_TRAIN / seq / "gt" / "gt.txt",
        IMAGE_TRAIN / seq / "gt" / "gt.txt",
    ]
    found = [p for p in candidates if p.is_file()]
    if not found:
        raise FileNotFoundError(
            f"No gt/gt.txt for {seq}. Checked: {candidates}. "
            "Expand the sequence folder in Kaggle and adjust this path if needed."
        )
    return found[0]

def frame_files(seq):
    """All frame image paths for a sequence, in frame order."""
    return sorted(
        (IMAGE_TRAIN / seq / "img1").glob("*.jpg"),
        key=lambda p: int(p.stem),
    )

def seq_image_size(seq):
    """(width, height) read from seqinfo.ini -- avoids opening every frame."""
    cfg = configparser.ConfigParser()
    cfg.read(IMAGE_TRAIN / seq / "seqinfo.ini")
    return int(cfg["Sequence"]["imWidth"]), int(cfg["Sequence"]["imHeight"])

for seq in SEQUENCES:
    frames = frame_files(seq)
    print(seq, "frames:", len(frames), "ground truth:", ground_truth_path(seq))
    if not frames:
        raise FileNotFoundError(f"No JPG frames in {IMAGE_TRAIN / seq / 'img1'}")

assert SEQUENCES, "No MOT20 training sequences found"
print("Images:", IMAGE_TRAIN)
print("Labels:", LABEL_TRAIN)

## Part 4: Shared Experiment Configuration
All hyper-parameters are defined **once** and used for both detectors, which is what makes the model-size comparison fair.

| Parameter | Value | Why |
| :--- | :--- | :--- |
| `IMGSZ` | 1280 | High resolution is needed to see small pedestrians in dense crowds |
| `CONF` | 0.10 | Deliberately low so ByteTrack keeps the **low-score detections** it uses in its second association step |
| `IOU` | 0.70 | NMS overlap threshold; kept high because people overlap heavily in MOT20 |
| `DEVICE` | GPU if available | Same hardware for all runs |

`CONF` and `IOU` are ByteTrack's operating point in Part 8. Detector-only accuracy in Part 6 uses Ultralytics' own confidence sweep.

In [ ]:
# Keep these IDENTICAL across yolo26s and yolo26m -- that's what makes the
# size comparison fair. CONF/IOU here are ByteTrack's operating point (used in
# the tracking loop, cell 6); detection-only accuracy in cell 5 uses Ultralytics'
# own internal PR-curve sweep instead of this single operating point.
MODELS = {
    "yolo26s": "yolo26s.pt",
    "yolo26m": "yolo26m.pt",
}

IMGSZ = 1280       # Keep identical for both models; reduce if GPU runs out of memory
CONF = 0.10        # Low enough to retain ByteTrack's lower-score detections
IOU = 0.70
DEVICE = 0 if torch.cuda.is_available() else "cpu"
HALF = torch.cuda.is_available()

def sync_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

## Part 5: Converting MOT20 Ground Truth to YOLO Format
MOT20's `gt.txt` is **not** in YOLO format, so the detector cannot be evaluated on it directly. This cell builds a parallel `images/` + `labels/` tree.

* `gt.txt` columns: `frame, id, x, y, w, h, mark, class, visibility`
* Only rows with `mark == 1` (considered for evaluation) and `class == 1` (pedestrian) are kept
* Boxes `[x, y, w, h]` in pixels become YOLO `[class, x_center, y_center, w, h]`, normalized to 0–1
* Frames are **symlinked**, not copied, to avoid duplicating several GB of images
* Everything goes into the `val` list, since the weights are pretrained and nothing is trained here, so there is no train/test leakage

In [ ]:
# MOT20's gt.txt is NOT YOLO format. We build a parallel images/labels tree
# using SYMLINKS (not copies) so we don't duplicate multi-GB image data.
# Every sequence goes into "val" here since these are off-the-shelf weights --
# there's no train split to leak from.

DET_DATA_DIR = OUT / "yolo_det_data"
DET_IMG_DIR = DET_DATA_DIR / "images"
DET_LBL_DIR = DET_DATA_DIR / "labels"
DET_DATA_DIR.mkdir(parents=True, exist_ok=True)

def convert_sequence_to_yolo(seq):
    width, height = seq_image_size(seq)
    gt = pd.read_csv(
        ground_truth_path(seq), header=None,
        names=["frame", "id", "x", "y", "w", "h", "mark", "cls", "vis"],
    )
    # MOT20: mark==1 means "consider for evaluation", cls==1 means pedestrian.
    gt = gt[(gt["mark"] == 1) & (gt["cls"] == 1)]
    gt_by_frame = {fid: rows for fid, rows in gt.groupby("frame")}

    img_out = DET_IMG_DIR / seq
    lbl_out = DET_LBL_DIR / seq
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    n = 0
    for frame_path in frame_files(seq):
        frame_id = int(frame_path.stem)
        link_path = img_out / frame_path.name
        if not link_path.exists():
            os.symlink(frame_path.resolve(), link_path)

        rows = gt_by_frame.get(frame_id)
        lines = []
        if rows is not None:
            for _, r in rows.iterrows():
                xc = (r["x"] + r["w"] / 2) / width
                yc = (r["y"] + r["h"] / 2) / height
                w = r["w"] / width
                h = r["h"] / height
                lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
        (lbl_out / f"{frame_id:06d}.txt").write_text("\n".join(lines))
        n += 1
    return n

for seq in SEQUENCES:
    count = convert_sequence_to_yolo(seq)
    print(f"{seq}: {count} frames converted to YOLO format")

val_images = sorted(str(p) for p in DET_IMG_DIR.glob("*/*.jpg"))
val_list_path = DET_DATA_DIR / "val_images.txt"
val_list_path.write_text("\n".join(val_images))

data_yaml_path = DET_DATA_DIR / "mot20.yaml"
data_yaml_path.write_text(
    # Ultralytics' check_det_dataset() requires a "train" key even when you only
    # call model.val() -- point it at the same file list as "val" since nothing
    # actually trains on it.
    f"path: {DET_DATA_DIR}\ntrain: val_images.txt\nval: val_images.txt\nnames:\n  0: person\n"
)
print(f"{len(val_images)} images ready for detection evaluation -> {data_yaml_path}")

## Part 6: Detector Benchmark (Detection Quality Only)
Runs `model.val()` for both detectors on all 8,931 frames and reports:

| Metric | Meaning |
| :--- | :--- |
| Precision | Of the boxes the detector drew, the fraction that were real people |
| Recall | Of the real people, the fraction the detector found |
| mAP50 | Average precision at IoU 0.50 |
| mAP50-95 | Average precision averaged over IoU 0.50–0.95 (stricter) |

This stage judges the **detector alone**, without tracking. It sets the upper bound for the tracker: a tracker cannot track a person the detector never found. Results are saved to `detection_metrics.csv`.

In [ ]:
detection_results = {}
for model_name, weights in MODELS.items():
    model = YOLO(weights)
    metrics = model.val(
        data=str(data_yaml_path),
        imgsz=IMGSZ,
        device=DEVICE,
        split="val",
        plots=False,
    )
    detection_results[model_name] = {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
    }
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

detection_df = pd.DataFrame(detection_results).T
detection_df.to_csv(OUT / "detection_metrics.csv")
display(detection_df)

## Part 7: ByteTrack Configuration
ByteTrack's key idea is to **use every detection box, including low-confidence ones**, instead of discarding them:

1. **First association:** high-score detections are matched to existing tracks (Kalman-predicted positions) using IoU and the Hungarian algorithm
2. **Second association:** low-score detections are matched to the tracks still unmatched, which recovers partly occluded people
3. Unmatched low-score boxes are discarded (they never start a new track); lost tracks are kept for a number of frames (`track_buffer`) before being deleted

ByteTrack uses **motion + IoU only**: no appearance (ReID) features and no camera-motion compensation, which keeps it very lightweight.

| Parameter | Role |
| :--- | :--- |
| `track_high_thresh` | Score above which a detection counts as "high" (first association) |
| `track_low_thresh` | Minimum score for the second association |
| `new_track_thresh` | Minimum score to start a new track |
| `track_buffer` | Frames a lost track is remembered |
| `match_thresh` | Maximum matching cost for an association |

We use the **default Ultralytics `bytetrack.yaml`**. The next cell prints the exact values used.

In [ ]:
from ultralytics.utils import ROOT

BYTETRACK_CFG = ROOT / "cfg" / "trackers" / "bytetrack.yaml"
print(f"Tracker configuration used by ByteTrack: {BYTETRACK_CFG}\n")
print(BYTETRACK_CFG.read_text())

## Part 8: Tracking Execution & System Profiling
For every detector and every sequence the loop:

1. Creates a **fresh `YOLO` instance** (this resets ByteTrack's state and IDs for each sequence)
2. Reads each frame and calls `model.track(..., tracker="bytetrack.yaml", persist=True)`
3. Writes tracks in **MOT Challenge format**: `frame, id, x, y, w, h, score, -1, -1, -1` → `mot20_benchmark/<model>/<sequence>.txt`
4. Times every frame

**What is timed:** image read + detection + tracking + transfer of results (end-to-end). The first frame of each sequence is excluded from the steady-state numbers as warm-up.

| Recorded field | Meaning |
| :--- | :--- |
| `fps` | Steady-state frames per second |
| `mean / p50 / p95_latency_ms` | Average, median and 95th-percentile per-frame time |
| `total_processing_s` | Total wall time for the sequence |
| `peak_gpu_allocated_mb` | Peak GPU memory allocated by PyTorch |

This is the longest cell of the notebook (8 runs = 2 models × 4 sequences). The per-sequence table is saved to `system_per_sequence.csv`.

In [ ]:
runs = []

for model_name, weights in MODELS.items():
    model_out = OUT / model_name
    model_out.mkdir(exist_ok=True)

    for seq in SEQUENCES:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        # A new instance resets ByteTrack IDs and state for each sequence.
        model = YOLO(weights)
        lines = []
        timed_ms = []
        all_ms = []

        frames = frame_files(seq)

        for index, frame_path in enumerate(frames):
            frame_id = int(frame_path.stem)

            sync_gpu()
            start = time.perf_counter()

            with Image.open(frame_path) as image:
                frame = np.asarray(image.convert("RGB"))[:, :, ::-1].copy()

            result = model.track(
                source=frame,
                persist=True,
                tracker="bytetrack.yaml",
                classes=[0],          # COCO person
                imgsz=IMGSZ,
                conf=CONF,
                iou=IOU,
                device=DEVICE,
                verbose=False,
            )[0]

            if result.boxes is not None and len(result.boxes):
                xyxy = result.boxes.xyxy.cpu().numpy()
                scores = result.boxes.conf.cpu().numpy()

                if result.boxes.id is not None:
                    ids = result.boxes.id.int().cpu().numpy()

                    for box, score, track_id in zip(xyxy, scores, ids):
                        x1, y1, x2, y2 = map(float, box)
                        lines.append(
                            f"{frame_id},{int(track_id)},"
                            f"{x1:.3f},{y1:.3f},"
                            f"{x2-x1:.3f},{y2-y1:.3f},"
                            f"{float(score):.6f},-1,-1,-1"
                        )

            sync_gpu()
            elapsed_ms = (time.perf_counter() - start) * 1000
            all_ms.append(elapsed_ms)
            if index > 0:
                timed_ms.append(elapsed_ms)

        (model_out / f"{seq}.txt").write_text(
            "\n".join(lines) + ("\n" if lines else "")
        )

        total_seconds = sum(all_ms) / 1000
        steady_seconds = sum(timed_ms) / 1000
        record = {
            "model": model_name,
            "sequence": seq,
            "frames": len(frames),
            "fps": len(timed_ms) / steady_seconds if steady_seconds else np.nan,
            "mean_latency_ms": np.mean(timed_ms) if timed_ms else np.nan,
            "p50_latency_ms": np.percentile(timed_ms, 50) if timed_ms else np.nan,
            "p95_latency_ms": np.percentile(timed_ms, 95) if timed_ms else np.nan,
            "total_processing_s": total_seconds,
            "peak_gpu_allocated_mb": (
                torch.cuda.max_memory_allocated() / 1024**2
                if torch.cuda.is_available() else np.nan
            ),
        }
        runs.append(record)
        print(record)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

system = pd.DataFrame(runs)
system.to_csv(OUT / "system_per_sequence.csv", index=False)
display(system)

## Part 9: Aggregated System Metrics
Combines the 4 per-sequence rows into **one row per detector**, saved to `system_metrics.csv`:

| Column | Aggregation |
| :--- | :--- |
| `fps` | Mean over sequences |
| `mean_latency_ms`, `p95_latency_ms` | Mean over sequences |
| `total_processing_s` | Sum over sequences |
| `peak_gpu_allocated_mb` | **Maximum** over sequences (true peak, not an average) |

In [ ]:

system_agg = (
    system.groupby("model")
    .agg(
        fps=("fps", "mean"),
        mean_latency_ms=("mean_latency_ms", "mean"),
        p95_latency_ms=("p95_latency_ms", "mean"),
        total_processing_s=("total_processing_s", "sum"),
        peak_gpu_allocated_mb=("peak_gpu_allocated_mb", "max"),  # true peak, not mean
    )
)
system_agg.to_csv(OUT / "system_metrics.csv")
display(system_agg)

## Part 10: Tracking Accuracy with TrackEval (HOTA / MOTA / IDF1)
Detection quality and ID consistency are graded separately. The tracking files from Part 8 are scored with the **official TrackEval** code on the pedestrian class of the MOT20 train sequences, combined over all 4 sequences.

| Metric | Full name | What it measures | Goal |
| :--- | :--- | :--- | :--- |
| **HOTA** | Higher Order Tracking Accuracy | Balanced score of detection and association quality | Higher ↑ |
| **DetA** | Detection Accuracy | How well people are detected (part of HOTA) | Higher ↑ |
| **AssA** | Association Accuracy | How well detections are linked into correct identities (part of HOTA) | Higher ↑ |
| **MOTA** | Multi-Object Tracking Accuracy | Missed people, false detections and ID switches in one score | Higher ↑ |
| **IDF1** | ID F1-Score | How consistently the same ID stays on the same person | Higher ↑ |
| **ID Switches** | – | Times a person's ID changed | Lower ↓ |
| **Frag** | Fragmentations | Times a trajectory was interrupted and resumed | Lower ↓ |

TrackEval expects `<tracker>/data/<sequence>.txt`, so the prediction files are **copied** (originals untouched) into `mot20_benchmark/trackeval_input/<model>/data/`. Results are saved to `tracking_metrics.csv`.

In [ ]:
import sys, shutil
import numpy as np
import pandas as pd
from pathlib import Path

# Use the TrackEval clone created in Part 2.
sys.path.insert(0, "/kaggle/working/TrackEval")

# NumPy compatibility fix (TrackEval uses NumPy aliases that were removed in recent versions)
if not hasattr(np, "asfarray"): np.asfarray = lambda a, *args, **kwargs: np.asarray(a, dtype=float)
if not hasattr(np, "float"): np.float = float
if not hasattr(np, "int"): np.int = int
if not hasattr(np, "bool"): np.bool = bool

import trackeval

def run_trackeval_hota(gt_folder, trackers_folder, tracker_name, seq_names):
    """Official HOTA / CLEAR / Identity via TrackEval."""
    eval_config = trackeval.Evaluator.get_default_eval_config()
    eval_config.update({
        "DISPLAY_LESS_PROGRESS": True,
        "PRINT_RESULTS": False,
        "USE_PARALLEL": False
    })

    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    dataset_config.update({
        "GT_FOLDER": str(gt_folder),
        "TRACKERS_FOLDER": str(trackers_folder),
        "OUTPUT_FOLDER": str(Path(trackers_folder) / "_trackeval_out"),
        "TRACKERS_TO_EVAL": [tracker_name],
        "CLASSES_TO_EVAL": ["pedestrian"],
        "BENCHMARK": "MOT20",
        "SPLIT_TO_EVAL": "train",
        "SKIP_SPLIT_FOL": True,
        "DO_PREPROC": True,
        "PRINT_CONFIG": False,
        "TRACKER_SUB_FOLDER": "data",
        "SEQ_INFO": {s: None for s in seq_names},
    })

    metrics = [trackeval.metrics.HOTA({"METRICS": ["HOTA"]}),
               trackeval.metrics.CLEAR({"METRICS": ["CLEAR"]}),
               trackeval.metrics.Identity({"METRICS": ["Identity"]})]

    evaluator = trackeval.Evaluator(eval_config)
    dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)
    out, _ = evaluator.evaluate([dataset], metrics)

    ped = out[list(out.keys())[0]][tracker_name]["COMBINED_SEQ"]["pedestrian"]

    def _mean(x):
        x = np.asarray(x, dtype=float)
        return float(np.mean(x))

    return {
        "HOTA (%)": round(_mean(ped["HOTA"]["HOTA"]) * 100, 1),
        "DetA (%)": round(_mean(ped["HOTA"]["DetA"]) * 100, 1),
        "AssA (%)": round(_mean(ped["HOTA"]["AssA"]) * 100, 1),
        "MOTA (%)": round(float(ped["CLEAR"]["MOTA"]) * 100, 1),
        "IDF1 (%)": round(float(ped["Identity"]["IDF1"]) * 100, 1),
        "ID Switches": int(ped["CLEAR"]["IDSW"]),
        "Frag": int(ped["CLEAR"]["Frag"]),
    }

# TrackEval wants <root>/<tracker>/data/<seq>.txt -> copy the ByteTrack outputs there.
EVAL_ROOT = OUT / "trackeval_input"
rows = []
for model_name in MODELS:
    dst = EVAL_ROOT / model_name / "data"
    dst.mkdir(parents=True, exist_ok=True)
    for seq in SEQUENCES:
        shutil.copy(OUT / model_name / f"{seq}.txt", dst / f"{seq}.txt")

    metrics = run_trackeval_hota(IMAGE_TRAIN, EVAL_ROOT, model_name, SEQUENCES)
    metrics["Model"] = model_name
    metrics["Tracker"] = "ByteTrack"
    rows.append(metrics)

bytetrack_tracking = pd.DataFrame(rows).set_index("Model")
bytetrack_tracking.to_csv(OUT / "tracking_metrics.csv")
display(bytetrack_tracking)

## Part 11: Visual Error Analysis (Qualitative Results)
Numbers hide *how* a tracker fails, so we look at it. For each selected frame the figure shows the **ground truth (left)** next to the **ByteTrack predictions (right)**.

* Every ID gets its own **stable colour**, so a person keeping the same colour across frames means the ID was kept
* The coloured line behind each box is a **15-frame trail** of that ID
* Predictions also show the detection score (e.g. `ID 12  0.87`)
* Three evenly spaced frames per sequence are shown

**What to look for:** people the tracker missed in dense regions, boxes on empty background, and an ID changing colour mid-trajectory (an ID switch).

### 11a. `yolo26s` + ByteTrack on MOT20-01

In [ ]:
# Cell 3A — Visualize tracking samples
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path

# Change these to inspect a different model or sequence.
VIS_MODEL = "yolo26s"                 # "yolo26s" or "yolo26m"
VIS_SEQUENCE = SEQUENCES[0]           # e.g. "MOT20-01"
NUM_SAMPLES = 3
TRAIL_LENGTH = 15                     # Number of preceding frames in each trail
DISPLAY_WIDTH = 1100                  # Width of each displayed panel

prediction_file = OUT / VIS_MODEL / f"{VIS_SEQUENCE}.txt"
if not prediction_file.exists():
    raise FileNotFoundError(
        f"{prediction_file} does not exist. Let Cell 3 finish this "
        "model and sequence before running the visualization."
    )

def read_mot_file(path):
    """Return rows grouped by frame; handle an empty prediction file."""
    grouped = defaultdict(list)
    for line in Path(path).read_text().splitlines():
        if not line.strip():
            continue
        values = [float(x) for x in line.split(",")]
        grouped[int(values[0])].append(values)
    return grouped

gt_by_frame = read_mot_file(ground_truth_path(VIS_SEQUENCE))
pred_by_frame = read_mot_file(prediction_file)

# MOT20 GT: column 7 is the annotation mark; column 8 is object class.
# Show valid pedestrians only.
gt_by_frame = {
    frame: [
        row for row in rows
        if len(row) >= 8 and int(row[6]) == 1 and int(row[7]) == 1
    ]
    for frame, rows in gt_by_frame.items()
}

def color_for_id(track_id):
    """Stable BGR color for an ID across frames."""
    rng = np.random.default_rng(int(track_id) + 12345)
    return tuple(int(x) for x in rng.integers(70, 256, size=3))

def draw_tracks(image, frame_number, rows_by_frame, show_scores=False):
    image = image.copy()
    height, width = image.shape[:2]
    thickness = max(2, round(width / 700))
    font_scale = max(0.45, width / 1800)

    current_rows = rows_by_frame.get(frame_number, [])

    for row in current_rows:
        track_id = int(row[1])
        x, y, box_w, box_h = row[2:6]
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + box_w), int(y + box_h)
        color = color_for_id(track_id)

        # Draw a trail using earlier observations of this same track ID.
        points = []
        first_frame = max(1, frame_number - TRAIL_LENGTH)
        for previous_frame in range(first_frame, frame_number + 1):
            matches = [
                r for r in rows_by_frame.get(previous_frame, [])
                if int(r[1]) == track_id
            ]
            if matches:
                r = matches[0]
                center_x = int(r[2] + r[4] / 2)
                bottom_y = int(r[3] + r[5])
                points.append((center_x, bottom_y))

        if len(points) >= 2:
            cv2.polylines(
                image,
                [np.array(points, dtype=np.int32)],
                False,
                color,
                thickness,
                cv2.LINE_AA,
            )

        cv2.rectangle(
            image, (x1, y1), (x2, y2),
            color, thickness, cv2.LINE_AA
        )

        label = f"ID {track_id}"
        if show_scores:
            label += f"  {row[6]:.2f}"

        (text_width, text_height), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
        )
        label_y = max(text_height + 8, y1)
        cv2.rectangle(
            image,
            (x1, label_y - text_height - 8),
            (x1 + text_width + 8, label_y + baseline),
            color, -1,
        )
        cv2.putText(
            image, label, (x1 + 4, label_y - 4),
            cv2.FONT_HERSHEY_SIMPLEX, font_scale,
            (0, 0, 0), thickness, cv2.LINE_AA,
        )

    header = f"Frame {frame_number} | {len(current_rows)} people"
    cv2.putText(
        image, header, (16, 34),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
        (255, 255, 255), 3, cv2.LINE_AA,
    )
    cv2.putText(
        image, header, (16, 34),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
        (0, 0, 0), 1, cv2.LINE_AA,
    )
    return image

def prepare_for_display(image):
    height, width = image.shape[:2]
    new_height = round(height * DISPLAY_WIDTH / width)
    resized = cv2.resize(image, (DISPLAY_WIDTH, new_height))
    return cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

available_frames = sorted(
    int(p.stem)
    for p in (IMAGE_TRAIN / VIS_SEQUENCE / "img1").glob("*.jpg")
)
if not available_frames:
    raise FileNotFoundError(f"No images found for {VIS_SEQUENCE}")

sample_indices = np.linspace(
    0, len(available_frames) - 1,
    min(NUM_SAMPLES, len(available_frames)),
    dtype=int,
)
sample_frames = [available_frames[i] for i in sample_indices]

for frame_number in sample_frames:
    image_path = (
        IMAGE_TRAIN / VIS_SEQUENCE / "img1" / f"{frame_number:06d}.jpg"
    )
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f"Could not read {image_path}")

    gt_image = draw_tracks(
        image, frame_number, gt_by_frame, show_scores=False
    )
    pred_image = draw_tracks(
        image, frame_number, pred_by_frame, show_scores=True
    )

    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    axes[0].imshow(prepare_for_display(gt_image))
    axes[0].set_title(f"{VIS_SEQUENCE} — Ground truth")
    axes[1].imshow(prepare_for_display(pred_image))
    axes[1].set_title(f"{VIS_MODEL} + ByteTrack — Predictions")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

### 11b. `yolo26m` + ByteTrack on MOT20-02
Same visualization with the larger detector on a longer sequence, to show how detector size changes the tracks.

In [ ]:
# Cell 3A — Visualize tracking samples
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path

# Change these to inspect a different model or sequence.
VIS_MODEL = "yolo26m"                 # "yolo26s" or "yolo26m"
VIS_SEQUENCE = SEQUENCES[1]           # e.g. "MOT20-01"
NUM_SAMPLES = 3
TRAIL_LENGTH = 15                     # Number of preceding frames in each trail
DISPLAY_WIDTH = 1100                  # Width of each displayed panel

prediction_file = OUT / VIS_MODEL / f"{VIS_SEQUENCE}.txt"
if not prediction_file.exists():
    raise FileNotFoundError(
        f"{prediction_file} does not exist. Let Cell 3 finish this "
        "model and sequence before running the visualization."
    )

def read_mot_file(path):
    """Return rows grouped by frame; handle an empty prediction file."""
    grouped = defaultdict(list)
    for line in Path(path).read_text().splitlines():
        if not line.strip():
            continue
        values = [float(x) for x in line.split(",")]
        grouped[int(values[0])].append(values)
    return grouped

gt_by_frame = read_mot_file(ground_truth_path(VIS_SEQUENCE))
pred_by_frame = read_mot_file(prediction_file)

# MOT20 GT: column 7 is the annotation mark; column 8 is object class.
# Show valid pedestrians only.
gt_by_frame = {
    frame: [
        row for row in rows
        if len(row) >= 8 and int(row[6]) == 1 and int(row[7]) == 1
    ]
    for frame, rows in gt_by_frame.items()
}

def color_for_id(track_id):
    """Stable BGR color for an ID across frames."""
    rng = np.random.default_rng(int(track_id) + 12345)
    return tuple(int(x) for x in rng.integers(70, 256, size=3))

def draw_tracks(image, frame_number, rows_by_frame, show_scores=False):
    image = image.copy()
    height, width = image.shape[:2]
    thickness = max(2, round(width / 700))
    font_scale = max(0.45, width / 1800)

    current_rows = rows_by_frame.get(frame_number, [])

    for row in current_rows:
        track_id = int(row[1])
        x, y, box_w, box_h = row[2:6]
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + box_w), int(y + box_h)
        color = color_for_id(track_id)

        # Draw a trail using earlier observations of this same track ID.
        points = []
        first_frame = max(1, frame_number - TRAIL_LENGTH)
        for previous_frame in range(first_frame, frame_number + 1):
            matches = [
                r for r in rows_by_frame.get(previous_frame, [])
                if int(r[1]) == track_id
            ]
            if matches:
                r = matches[0]
                center_x = int(r[2] + r[4] / 2)
                bottom_y = int(r[3] + r[5])
                points.append((center_x, bottom_y))

        if len(points) >= 2:
            cv2.polylines(
                image,
                [np.array(points, dtype=np.int32)],
                False,
                color,
                thickness,
                cv2.LINE_AA,
            )

        cv2.rectangle(
            image, (x1, y1), (x2, y2),
            color, thickness, cv2.LINE_AA
        )

        label = f"ID {track_id}"
        if show_scores:
            label += f"  {row[6]:.2f}"

        (text_width, text_height), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
        )
        label_y = max(text_height + 8, y1)
        cv2.rectangle(
            image,
            (x1, label_y - text_height - 8),
            (x1 + text_width + 8, label_y + baseline),
            color, -1,
        )
        cv2.putText(
            image, label, (x1 + 4, label_y - 4),
            cv2.FONT_HERSHEY_SIMPLEX, font_scale,
            (0, 0, 0), thickness, cv2.LINE_AA,
        )

    header = f"Frame {frame_number} | {len(current_rows)} people"
    cv2.putText(
        image, header, (16, 34),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
        (255, 255, 255), 3, cv2.LINE_AA,
    )
    cv2.putText(
        image, header, (16, 34),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
        (0, 0, 0), 1, cv2.LINE_AA,
    )
    return image

def prepare_for_display(image):
    height, width = image.shape[:2]
    new_height = round(height * DISPLAY_WIDTH / width)
    resized = cv2.resize(image, (DISPLAY_WIDTH, new_height))
    return cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

available_frames = sorted(
    int(p.stem)
    for p in (IMAGE_TRAIN / VIS_SEQUENCE / "img1").glob("*.jpg")
)
if not available_frames:
    raise FileNotFoundError(f"No images found for {VIS_SEQUENCE}")

sample_indices = np.linspace(
    0, len(available_frames) - 1,
    min(NUM_SAMPLES, len(available_frames)),
    dtype=int,
)
sample_frames = [available_frames[i] for i in sample_indices]

for frame_number in sample_frames:
    image_path = (
        IMAGE_TRAIN / VIS_SEQUENCE / "img1" / f"{frame_number:06d}.jpg"
    )
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f"Could not read {image_path}")

    gt_image = draw_tracks(
        image, frame_number, gt_by_frame, show_scores=False
    )
    pred_image = draw_tracks(
        image, frame_number, pred_by_frame, show_scores=True
    )

    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    axes[0].imshow(prepare_for_display(gt_image))
    axes[0].set_title(f"{VIS_SEQUENCE} — Ground truth")
    axes[1].imshow(prepare_for_display(pred_image))
    axes[1].set_title(f"{VIS_MODEL} + ByteTrack — Predictions")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Part 12: Tracking Video Demos
Renders the first 300 frames of **MOT20-02** into an MP4 with boxes coloured by track ID and labelled `ID  score`, at the sequence's real frame rate. One video per detector.

> If the inline player is blank, the video is still saved under `mot20_benchmark/videos/`. Download it from the **Output** tab (the `mp4v` codec is not supported by every browser).

### 12a. `yolo26m` + ByteTrack

In [ ]:
import cv2
import configparser
from pathlib import Path
from IPython.display import Video, display

MODEL_NAME = "yolo26m"       # Change to "yolo26s" for the other model
SEQUENCE = "MOT20-02"
START_FRAME = 1
MAX_FRAMES = 300             # Set to None for the complete sequence
OUTPUT_WIDTH = 1280          # Smaller video; set to None for original size

image_dir = IMAGE_TRAIN / SEQUENCE / "img1"
tracking_file = OUT / MODEL_NAME / f"{SEQUENCE}.txt"
output_dir = OUT / "videos"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f"{MODEL_NAME}_{SEQUENCE}.mp4"

if not tracking_file.exists():
    raise FileNotFoundError(
        f"{tracking_file} is missing. Finish Cell 3 for this model first."
    )

# Load ByteTrack predictions, grouped by frame number.
tracks_by_frame = {}
for line in tracking_file.read_text().splitlines():
    if not line.strip():
        continue
    row = [float(value) for value in line.split(",")]
    tracks_by_frame.setdefault(int(row[0]), []).append(row)

# Use the sequence's actual frame rate.
config = configparser.ConfigParser()
config.read(IMAGE_TRAIN / SEQUENCE / "seqinfo.ini")
fps = float(config["Sequence"]["frameRate"])

frames = sorted(image_dir.glob("*.jpg"), key=lambda p: int(p.stem))
frames = [p for p in frames if int(p.stem) >= START_FRAME]
if MAX_FRAMES is not None:
    frames = frames[:MAX_FRAMES]

if not frames:
    raise ValueError("No frames selected. Check START_FRAME and SEQUENCE.")

first = cv2.imread(str(frames[0]))
if first is None:
    raise RuntimeError(f"Cannot read {frames[0]}")

source_height, source_width = first.shape[:2]
video_width = OUTPUT_WIDTH or source_width
video_height = round(source_height * video_width / source_width)
scale = video_width / source_width

writer = cv2.VideoWriter(
    str(output_file),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (video_width, video_height),
)
if not writer.isOpened():
    raise RuntimeError("Could not create MP4 video")

def id_color(track_id):
    # Consistent color for an ID throughout the video (BGR).
    return (
        (track_id * 53 + 80) % 256,
        (track_id * 97 + 100) % 256,
        (track_id * 139 + 120) % 256,
    )

try:
    for index, image_path in enumerate(frames):
        frame_number = int(image_path.stem)
        frame = cv2.imread(str(image_path))
        if frame is None:
            raise RuntimeError(f"Cannot read {image_path}")

        frame = cv2.resize(frame, (video_width, video_height))

        for row in tracks_by_frame.get(frame_number, []):
            track_id = int(row[1])
            x, y, box_width, box_height = row[2:6]
            score = row[6]

            x1 = round(x * scale)
            y1 = round(y * scale)
            x2 = round((x + box_width) * scale)
            y2 = round((y + box_height) * scale)
            color = id_color(track_id)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            label = f"ID {track_id}  {score:.2f}"
            cv2.putText(
                frame, label,
                (x1, max(22, y1 - 7)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55, color, 2, cv2.LINE_AA,
            )

        cv2.putText(
            frame, f"{MODEL_NAME} + ByteTrack | {SEQUENCE} | frame {frame_number}",
            (15, 32),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75, (255, 255, 255), 2, cv2.LINE_AA,
        )
        writer.write(frame)

        if (index + 1) % 100 == 0:
            print(f"Written {index + 1}/{len(frames)} frames")
finally:
    writer.release()

print("Video saved to:", output_file)
display(Video(str(output_file), embed=True, width=960))

### 12b. `yolo26s` + ByteTrack

In [ ]:
import cv2
import configparser
from pathlib import Path
from IPython.display import Video, display

MODEL_NAME = "yolo26s"       # Change to "yolo26s" for the other model
SEQUENCE = "MOT20-02"
START_FRAME = 1
MAX_FRAMES = 300             # Set to None for the complete sequence
OUTPUT_WIDTH = 1280          # Smaller video; set to None for original size

image_dir = IMAGE_TRAIN / SEQUENCE / "img1"
tracking_file = OUT / MODEL_NAME / f"{SEQUENCE}.txt"
output_dir = OUT / "videos"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f"{MODEL_NAME}_{SEQUENCE}.mp4"

if not tracking_file.exists():
    raise FileNotFoundError(
        f"{tracking_file} is missing. Finish Cell 3 for this model first."
    )

# Load ByteTrack predictions, grouped by frame number.
tracks_by_frame = {}
for line in tracking_file.read_text().splitlines():
    if not line.strip():
        continue
    row = [float(value) for value in line.split(",")]
    tracks_by_frame.setdefault(int(row[0]), []).append(row)

# Use the sequence's actual frame rate.
config = configparser.ConfigParser()
config.read(IMAGE_TRAIN / SEQUENCE / "seqinfo.ini")
fps = float(config["Sequence"]["frameRate"])

frames = sorted(image_dir.glob("*.jpg"), key=lambda p: int(p.stem))
frames = [p for p in frames if int(p.stem) >= START_FRAME]
if MAX_FRAMES is not None:
    frames = frames[:MAX_FRAMES]

if not frames:
    raise ValueError("No frames selected. Check START_FRAME and SEQUENCE.")

first = cv2.imread(str(frames[0]))
if first is None:
    raise RuntimeError(f"Cannot read {frames[0]}")

source_height, source_width = first.shape[:2]
video_width = OUTPUT_WIDTH or source_width
video_height = round(source_height * video_width / source_width)
scale = video_width / source_width

writer = cv2.VideoWriter(
    str(output_file),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (video_width, video_height),
)
if not writer.isOpened():
    raise RuntimeError("Could not create MP4 video")

def id_color(track_id):
    # Consistent color for an ID throughout the video (BGR).
    return (
        (track_id * 53 + 80) % 256,
        (track_id * 97 + 100) % 256,
        (track_id * 139 + 120) % 256,
    )

try:
    for index, image_path in enumerate(frames):
        frame_number = int(image_path.stem)
        frame = cv2.imread(str(image_path))
        if frame is None:
            raise RuntimeError(f"Cannot read {image_path}")

        frame = cv2.resize(frame, (video_width, video_height))

        for row in tracks_by_frame.get(frame_number, []):
            track_id = int(row[1])
            x, y, box_width, box_height = row[2:6]
            score = row[6]

            x1 = round(x * scale)
            y1 = round(y * scale)
            x2 = round((x + box_width) * scale)
            y2 = round((y + box_height) * scale)
            color = id_color(track_id)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            label = f"ID {track_id}  {score:.2f}"
            cv2.putText(
                frame, label,
                (x1, max(22, y1 - 7)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55, color, 2, cv2.LINE_AA,
            )

        cv2.putText(
            frame, f"{MODEL_NAME} + ByteTrack | {SEQUENCE} | frame {frame_number}",
            (15, 32),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75, (255, 255, 255), 2, cv2.LINE_AA,
        )
        writer.write(frame)

        if (index + 1) % 100 == 0:
            print(f"Written {index + 1}/{len(frames)} frames")
finally:
    writer.release()

print("Video saved to:", output_file)
display(Video(str(output_file), embed=True, width=960))

## Part 13: Conclusions & Limitations

### Key findings
* **Tracking accuracy:** ByteTrack reaches HOTA = [x] with `yolo26s` and [y] with `yolo26m` (Part 10).
* **Identity consistency:** IDF1 = [x] / [y] with [n] / [n] ID switches for `yolo26s` / `yolo26m`.
* **Detector size:** moving from `yolo26s` to `yolo26m` changes HOTA by [+x], mostly through **DetA** (detection) rather than **AssA** (association), which shows the detector is the main bottleneck.
* **Cost:** ByteTrack runs at [x] FPS end-to-end with `yolo26s` and [x] FPS with `yolo26m`, using at most [x] MB of GPU memory (Part 9).
* **Behaviour in crowds:** the visualizations (Part 11) show [missed people in dense regions / ID switches after occlusion / ...].

### Limitations
* Detectors are **off-the-shelf COCO weights**, not fine-tuned on MOT20, so recall in dense crowds is the main limitation (see Part 6).
* Evaluation uses the **train split** because MOT20 test ground truth is hidden.
* ByteTrack runs with its **default configuration**; no hyper-parameter tuning was done.
* Results come from a single run on one GPU type (Tesla T4).

### Future work
Fine-tune the detectors on MOT20 and tune ByteTrack's thresholds and `track_buffer` for dense scenes.